# ParPTE · Speaking model training (pronunciation, fluency, prosody)

ParPTE **already ships a trained speaking model** (`backend/models/speaking_traits.joblib`). Use this notebook to re-train it after you change the feature extractor, or to add Part B.

**Data:** [speechocean762](https://github.com/jimbozhang/speechocean762): 5,000 utterances from non-native English speakers, each rated by five experts for pronunciation accuracy, fluency, prosody and completeness (0–10). The dataset's README states it is free for commercial and non-commercial use.

| Part | What it trains | Hardware | Time |
|---|---|---|---|
| A | Gradient-boosted trait models on ParPTE's acoustic features (word-level GOP from PocketSphinx alignment, Silero VAD pauses, syllable rate, fillers, pitch) | CPU | ≈10 min on Kaggle's 4 CPUs |
| B (optional) | Fine-tuned `wav2vec2-base` on raw audio → ONNX, blended in by the server | GPU T4 | ≈1.5 h |

Reference results (held-out test split): Part A reaches PCC ≈ 0.67 accuracy / 0.75 fluency / 0.74 prosody. The published GOPT research model reports ≈ 0.71 / 0.75 / 0.76.

**Settings:** Internet **On**. GPU only needed for Part B. Then *Run all*, and download the files from **Output**.

In [ ]:
import os, sys, subprocess
REPO = "https://github.com/chiran-commits/ParPTE.git"
try:  # private repo: add a read-only GitHub token as a Kaggle secret named GITHUB_TOKEN
    from kaggle_secrets import UserSecretsClient
    REPO = REPO.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("No GITHUB_TOKEN secret — cloning as a public repo")
W = "/kaggle/working"
if not os.path.exists(f"{W}/ParPTE"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", REPO, f"{W}/ParPTE"], check=True)
if not os.path.exists(f"{W}/speechocean762"):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/jimbozhang/speechocean762", f"{W}/speechocean762"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{W}/ParPTE/backend/requirements.txt", "pandas"], check=True)
sys.path.insert(0, f"{W}/ParPTE/backend")
print("ready")

## Part A · 1 — Extract features (same code the server runs)

In [ ]:
os.chdir(f"{W}/ParPTE/backend")
subprocess.run([sys.executable, "training/extract_speechocean.py", f"{W}/speechocean762", f"{W}/features.csv", "--workers", str(os.cpu_count())], check=True)

## Part A · 2 — Train and evaluate on the held-out test split

In [ ]:
os.makedirs(f"{W}/parpte_models", exist_ok=True)
subprocess.run([sys.executable, "training/train_speaking.py", f"{W}/features.csv", f"{W}/parpte_models/speaking_traits.joblib"], check=True)

## Part B (optional) · fine-tune wav2vec2 on raw audio
Set `RUN_PART_B = True` with a GPU attached. The server blends this model with Part A when `speaking_ssl/model.onnx` is present.

In [ ]:
RUN_PART_B = False

if RUN_PART_B:
    import json, random, wave
    import numpy as np, torch, torch.nn as nn
    from scipy.stats import pearsonr
    from tqdm.auto import tqdm
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "onnx"], check=True)
    from transformers import Wav2Vec2Model

    root = f"{W}/speechocean762"
    scores = json.load(open(f"{root}/resource/scores.json"))
    TARGETS, MAX_S, device = ["accuracy", "fluency", "prosodic"], 15, "cuda"

    def load(split):
        out = []
        for line in open(f"{root}/{split}/wav.scp"):
            utt, rel = line.split()
            with wave.open(f"{root}/{rel}") as w:
                x = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(np.float32)[: 16000 * MAX_S] / 32768
            out.append(((x - x.mean()) / (x.std() + 1e-7), np.array([scores[utt][t] for t in TARGETS], dtype=np.float32)))
        return out
    train, test = load("train"), load("test")

    class Scorer(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
            self.enc.feature_extractor._freeze_parameters()
            self.head = nn.Sequential(nn.Linear(768, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 3))
        def forward(self, x):
            return self.head(self.enc(x).last_hidden_state.mean(1))

    def batches(data, bs, shuffle):
        idx = list(range(len(data)))
        if shuffle: random.shuffle(idx)
        for i in range(0, len(idx), bs):
            items = [data[j] for j in idx[i:i + bs]]
            L = max(len(w) for w, _ in items)
            yield (torch.from_numpy(np.stack([np.pad(w, (0, L - len(w))) for w, _ in items])).to(device),
                   torch.from_numpy(np.stack([t for _, t in items])).to(device))

    model = Scorer().to(device)
    opt = torch.optim.AdamW([{"params": model.enc.parameters(), "lr": 3e-5}, {"params": model.head.parameters(), "lr": 1e-3}], weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler()
    for ep in range(6):
        model.train()
        for x, y in tqdm(batches(train, 8, True), total=len(train) // 8, desc=f"epoch {ep + 1}"):
            with torch.autocast("cuda", dtype=torch.float16):
                loss = nn.functional.mse_loss(model(x).float(), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        model.eval(); P, Y = [], []
        with torch.no_grad():
            for x, y in batches(test, 1, False):
                P.append(model(x).float().cpu().numpy()[0]); Y.append(y.cpu().numpy()[0])
        P, Y = np.array(P), np.array(Y)
        print(f"epoch {ep + 1}: " + "  ".join(f"{t} PCC={pearsonr(P[:, i], Y[:, i])[0]:.3f}" for i, t in enumerate(TARGETS)))

    out = f"{W}/parpte_models/speaking_ssl"
    os.makedirs(out, exist_ok=True)
    model = model.float().cpu().eval()
    torch.onnx.export(model, torch.randn(1, 16000 * 4), f"{out}/model.onnx", input_names=["input_values"], output_names=["scores"],
                      dynamic_axes={"input_values": {0: "batch", 1: "samples"}, "scores": {0: "batch"}}, opset_version=17, dynamo=False)
    json.dump({"targets": TARGETS, "max_seconds": 30, "base_model": "facebook/wav2vec2-base",
               "pcc": {t: float(pearsonr(P[:, i], Y[:, i])[0]) for i, t in enumerate(TARGETS)}}, open(f"{out}/config.json", "w"), indent=2)

In [ ]:
import shutil
shutil.make_archive(f"{W}/parpte_speaking_models", "zip", f"{W}/parpte_models")
print("Download parpte_speaking_models.zip and unzip it into backend/models/")